In [ ]:
# ==========================================================
#                 QUICKNOTE ML — CLEAN VERSION
#            PDF → RAG → OPENROUTER GEMMA-3-4B-IT
# ==========================================================

# Install required packages
!pip install -q chromadb sentence-transformers pdfplumber graphviz requests

# Imports
import os, sys, time, re, json, subprocess, textwrap
import pdfplumber, chromadb, requests
from typing import List
from sentence_transformers import SentenceTransformer
from IPython.display import Image, display

# ----------------------------------------------------------
# CHECK GRAPHVIZ
# ----------------------------------------------------------
GRAPHVIZ_AVAILABLE = False
try:
    result = subprocess.run(['dot', '-V'], capture_output=True, text=True)
    if result.returncode == 0:
        GRAPHVIZ_AVAILABLE = True
        print("✅ Graphviz detected — Visual mindmaps available")
    else:
        print("⚠️ Graphviz not found — install from graphviz.org for visual mindmaps")
except:
    print("⚠️ Graphviz not installed")

# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------
MODEL = "google/gemma-3-4b-it"
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MAX_RETRIES = 3
RETRY_DELAY = 2

# ----------------------------------------------------------
# API KEY INPUT (SECURE - NO KEY SHOWN)
# ----------------------------------------------------------
print("Enter Your Api Key : (openrouter key)")
OPENROUTER_API_KEY = input().strip()

if len(OPENROUTER_API_KEY) < 20:
    raise ValueError("❌ Invalid API Key")

# Validate API key
try:
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    test = requests.post(API_URL, headers=headers,
                         json={"model": MODEL,
                               "messages":[{"role":"user","content":"Hi"}],
                               "max_tokens":5},
                         timeout=10)
    if test.status_code == 401:
        raise ValueError("❌ Authentication failed — Wrong API key")
    print("✅ API Key verified")
except Exception as e:
    print("⚠️ API Test Warning:", e)

# ----------------------------------------------------------
# LOAD EMBEDDING MODEL + VECTOR DB
# ----------------------------------------------------------
print("🔄 Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()

try: client.delete_collection("pdf_rag")
except: pass
collection = client.create_collection("pdf_rag")
print("✅ Vector DB Ready")

# ----------------------------------------------------------
# PDF PROCESSING
# ----------------------------------------------------------
def extract_pdf_text(path: str) -> str:
    text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t: text.append(t)
    out = "\n\n".join(text)
    if len(out) < 100:
        raise ValueError("PDF too short or scanned.")
    return out

def chunk_text(text: str, size=900, overlap=200) -> List[str]:
    chunks, start = [], 0
    step = size - overlap
    while start < len(text):
        chunk = text[start:start+size].strip()
        if len(chunk) > 50: chunks.append(chunk)
        start += step
    return chunks

def add_chunks(chunks: List[str]):
    if not chunks: return
    print(f"🔄 Embedding {len(chunks)} chunks...")
    vectors = embedder.encode(chunks, show_progress_bar=True).tolist()
    ids = [f"chunk-{i}" for i in range(len(chunks))]
    collection.add(ids=ids, documents=chunks, embeddings=vectors)
    print(f"✅ Stored {len(chunks)} chunks")

def get_context(query: str, k: int = 8):
    if collection.count() == 0:
        return "[NO DATA]"
    q = embedder.encode(query).tolist()
    result = collection.query(query_embeddings=[q], n_results=k)
    return "\n\n---\n\n".join(result["documents"][0])

# ----------------------------------------------------------
# OPENROUTER CALL
# ----------------------------------------------------------
def call_llm(prompt: str, temp=0.7, max_tokens=2000):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [{"role":"user","content":prompt}],
        "temperature": temp,
        "max_tokens": max_tokens
    }

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(API_URL, headers=headers, json=payload, timeout=60)
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"]
            elif r.status_code == 429:
                time.sleep(RETRY_DELAY * (attempt+1))
            else:
                time.sleep(RETRY_DELAY)
        except:
            time.sleep(RETRY_DELAY)
    return "❌ API Error"

# ----------------------------------------------------------
# GENERATORS
# ----------------------------------------------------------
def generate_summary():
    ctx = get_context("summary main concepts", 10)
    prompt = f"""
Write a 350–400 word academic summary. Use ONLY this context:

Context:
{ctx}

Summary:
"""
    return call_llm(prompt, 0.5, 600)

def generate_five_mark(topic):
    ctx = get_context(topic, 6)
    prompt = f"""
Write a 5-mark answer (10–11 lines, 120–150 words, academic tone) for:

Topic: {topic}

Context:
{ctx}
"""
    return call_llm(prompt, 0.4, 300)

def generate_twelve_mark(topic):
    ctx = get_context(topic, 8)
    prompt = f"""
Write a 12-mark answer (250–300 words) AND include this ASCII flowchart:

Start
 |
 v
[Step 1]
 |
 v
{{{{Decision?}}}}
 /   \\
Yes   No
 |     |
 v     v
[Path A]   [Path B]
 |
 v
End

Use ONLY this context:

{ctx}
"""
    return call_llm(prompt, 0.5, 900)

def generate_ascii_mindmap(topic):
    ctx = get_context(topic, 5)
    prompt = f"""
Create an ASCII mindmap for topic "{topic}". Use hierarchical structure only.

Context:
{ctx}
"""
    return call_llm(prompt, 0.6, 500)

def generate_visual_mindmap(topic):
    if not GRAPHVIZ_AVAILABLE:
        return None

    ctx = get_context(topic, 6)
    prompt = f"""
Generate VALID Graphviz DOT code for a mindmap about "{topic}".
Return ONLY DOT code. It must begin with 'digraph' and end with '}}'.

Context:
{ctx}
"""
    dot = call_llm(prompt, 0.3, 800)

    # Extract code without markdown
    if "```" in dot:
        dot = re.sub(r"```.*?```", "", dot, flags=re.DOTALL).strip()

    if not dot.startswith("digraph"):
        return None

    # Fix unbalanced braces if needed
    if dot.count("{") > dot.count("}"):
        dot += "}" * (dot.count("{") - dot.count("}"))

    return dot

# ----------------------------------------------------------
# PDF UPLOAD
# ----------------------------------------------------------
def upload_pdf():
    print("\n📁 Enter PDF path:")
    path = input("Path: ").strip().strip('"').strip("'")

    if not os.path.exists(path):
        print("❌ File not found")
        return False
    if not path.lower().endswith(".pdf"):
        print("❌ Not a PDF")
        return False

    print("📄 Extracting PDF...")
    text = extract_pdf_text(path)
    chunks = chunk_text(text)
    add_chunks(chunks)
    print(f"✅ Loaded {len(chunks)} chunks")
    return True

# ----------------------------------------------------------
# INTERFACE
# ----------------------------------------------------------
def menu():
    print("\n" + "="*50)
    print("📚 COMMANDS")
    print("summary")
    print("5 mark <topic>")
    print("12 mark <topic>")
    print("mindmap <topic>")
    print("visual <topic>")
    print("ask <question>")
    print("bye")
    print("="*50)

def main():
    print("🚀 QUICKNOTE ML (CLEAN VERSION)")
    print("="*50)

    if not upload_pdf():
        print("❌ Failed to load PDF")
        return

    menu()

    while True:
        user = input("\nYou: ").strip()
        low = user.lower()

        if low == "bye":
            print("👋 Goodbye!")
            break

        if low == "summary":
            print(generate_summary()); continue

        if low.startswith("5 mark "):
            print(generate_five_mark(user[7:])); continue

        if low.startswith("12 mark "):
            print(generate_twelve_mark(user[8:])); continue

        if low.startswith("mindmap "):
            print(generate_ascii_mindmap(user[8:])); continue

        if low.startswith("visual "):
            topic = user[7:]
            dot = generate_visual_mindmap(topic)
            if dot:
                src = Source(dot)
                src.format = "png"
                fname = f"mindmap_{int(time.time())}"
                src.render(fname, cleanup=True)
                display(Image(filename=f"{fname}.png"))
            else:
                print("❌ Failed to generate visual mindmap")
            continue

        if low.startswith("ask "):
            q = user[4:]
            ctx = get_context(q, 5)
            print(call_llm(f"Use ONLY context:\n{ctx}\n\nAnswer: {q}"))
            continue

        print("❓ Unknown command")

main()